In [ ]:
import torch
print("GPU disponível:", torch.cuda.is_available())
print("Modelo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma")


GPU disponível: True
Modelo: Tesla T4


In [3]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;


In [8]:
!git clone https://github.com/annaferreiras/ft-llm-tool-calling.git

from datasets import load_dataset

ds = load_dataset("json", data_files={
    "train":      "ft-llm-tool-calling/dados/treino.jsonl",
    "validation": "ft-llm-tool-calling/dados/validacao.jsonl",
    "test":       "ft-llm-tool-calling/dados/teste.jsonl",
})
print(ds)

Cloning into 'ft-llm-tool-calling'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 26 (delta 8), reused 24 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 235.90 KiB | 939.00 KiB/s, done.
Resolving deltas: 100% (8/8), done.


DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 386
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 47
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 51
    })
})


In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name      = "unsloth/gemma-4-E2B-it",
    dtype           = None,
    max_seq_length  = 2048,
    load_in_4bit    = True, #vai fazer o modelo de 5,12B caber em 16GB
    full_finetuning = False, #usar LoRA
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.6: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


In [ ]:
textos = [tokenizer.apply_chat_template(ex["messages"], tokenize=False) for ex in ds["train"]]

tk = getattr(tokenizer, "tokenizer", tokenizer)   # o tokenizer de texto que vive dentro do processor
tamanhos = sorted(len(tk(t)["input_ids"]) for t in textos)

n = len(tamanhos)
print("mediana:", tamanhos[n//2])
print("p90:    ", tamanhos[int(n*0.9)])
print("maximo: ", tamanhos[-1])
print("acima de 2048:", sum(1 for t in tamanhos if t > 2048))

mediana: 813
p90:     1083
maximo:  1549
acima de 2048: 0


#### Configurar LoRA

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r            = 8, #largura das matrizes treinadas
    lora_alpha   = 8,
    lora_dropout = 0,
    bias         = "none",
    random_state = 3407,
)
model.print_trainable_parameters()


In [ ]:
def formatar(exemplos):
    textos = [
        tokenizer.apply_chat_template(msgs, tokenize=False).removeprefix("<bos>")
        for msgs in exemplos["messages"]
    ]
    return {"text": textos}

ds = ds.map(formatar, batched=True)
print(ds["train"][0]["text"][:700])

<|turn>system
<instrucoes>
Você é um assistente de IA com acesso a um conjunto de ferramentas.
Seu objetivo é responder às perguntas do usuário de forma correta e útil.

Como proceder:

1. Analise o pedido do usuário e entenda a intenção.
2. Decida se precisa de ferramenta:
   - Se você consegue responder com o que já sabe, responda direto.
   - Se precisa de informação externa ou de executar uma ação, use uma ferramenta.
3. Se for usar uma ferramenta, escolha a mais adequada entre as disponíveis
   e extraia os argumentos a partir do pedido do usuário.
4. Para chamar uma ferramenta, escreva a chamada dentro de <tool_call>.
   O conteúdo deve ser um objeto JSON:

   <tool_call>
   {"nome_too


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = ds["train"],
    eval_dataset  = ds["validation"],
    args = SFTConfig(
        dataset_text_field = "text",
        gradient_checkpointing = "unsloth",


        per_device_train_batch_size = 1,
        per_device_eval_batch_size  = 1,
        gradient_accumulation_steps = 8,

        num_train_epochs = 3,
        learning_rate    = 2e-4,
        warmup_ratio     = 0.03,
        max_grad_norm    = 0.3,
        weight_decay     = 0.001,
        lr_scheduler_type = "linear",
        optim = "adamw_8bit",

        logging_steps = 5,
        eval_strategy = "steps",
        eval_steps    = 10,
        save_strategy = "steps",
        save_steps    = 10,
        load_best_model_at_end = True,
        metric_for_best_model  = "eval_loss",
        greater_is_better      = False,

        output_dir = "outputs",
        seed = 3407,
        report_to = "none",
    ),
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Switching to float32 training since model cannot work with float16


In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(trainer)

Unsloth: Auto-detected instruction_part = '<|turn>user\n' and response_part = '<|turn>model\n'


In [ ]:
tk = getattr(tokenizer, "tokenizer", tokenizer)

exemplo = trainer.train_dataset[0]
print(tk.decode(
    [tk.pad_token_id if x == -100 else x for x in exemplo["labels"]]
).replace(tk.pad_token, " "))


                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  <final_answer>
Um indivíduo que resolve problemas de forma eficaz geralmente possui habilidades analíticas fortes, pensamento crítico e capacidade de abordar desafios de diferentes ângulos. Além disso, eles costumam ser pacientes, persistentes e flexíveis, o que lhes permite adaptar suas estratégias conforme necessário. Essas características permitem que eles identifiquem soluções inovadoras e eficientes para os problemas que enfrentam.
</final_answer><turn|>



In [ ]:
import torch
gpu = torch.cuda.get_device_properties(0)
total = round(gpu.total_memory / 1024**3, 2)
usada = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f"GPU: {gpu.name} | total: {total} GB | já reservada: {usada} GB")

GPU: Tesla T4 | total: 14.56 GB | já reservada: 12.01 GB


In [ ]:
import torch
print("em uso:    ", round(torch.cuda.memory_allocated()/1024**3, 2), "GB")
print("reservada: ", round(torch.cuda.memory_reserved()/1024**3, 2), "GB")

em uso:     7.63 GB
reservada:  7.72 GB


In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 386 | Num Epochs = 3 | Total steps = 147
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 12,668,928 of 5,135,846,944 (0.25% trained)


Step,Training Loss,Validation Loss
10,0.149792,0.939093
20,0.089045,0.646226
30,0.077280,0.593627
40,0.081639,0.545749
50,0.115101,0.528203
60,0.058817,0.523526
70,0.060510,0.513548
80,0.074269,0.504501
90,0.065765,0.497667
100,0.071903,0.492359


Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-10/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-20/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-30/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-40/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-70/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-80/tokenizer_con

In [ ]:
model.save_pretrained("gemma4_tool_lora")
tokenizer.save_pretrained("gemma4_tool_lora")

# conferir o que foi salvo e o tamanho
!du -sh gemma4_tool_lora
!ls -la gemma4_tool_lora

Unsloth: Restored added_tokens_decoder metadata in gemma4_tool_lora/tokenizer_config.json.


80M	gemma4_tool_lora
total 81028
drwxr-xr-x 2 root root     4096 Aug  1 02:43 .
drwxr-xr-x 1 root root     4096 Aug  1 02:43 ..
-rw-r--r-- 1 root root     1657 Aug  1 02:43 adapter_config.json
-rw------- 1 root root 50747584 Aug  1 02:43 adapter_model.safetensors
-rw-r--r-- 1 root root    18810 Aug  1 02:43 chat_template.jinja
-rw-r--r-- 1 root root     1689 Aug  1 02:43 processor_config.json
-rw-r--r-- 1 root root     5254 Aug  1 02:43 README.md
-rw-r--r-- 1 root root     6864 Aug  1 02:43 tokenizer_config.json
-rw-r--r-- 1 root root 32169626 Aug  1 02:43 tokenizer.json


In [ ]:
!zip -r -q gemma4_tool_lora.zip gemma4_tool_lora

from google.colab import files
files.download("gemma4_tool_lora.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json

def responder(mensagens, max_novos_tokens=300):
    texto = tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True,
    )

    tk = getattr(tokenizer, "tokenizer", tokenizer)
    entrada = tk(texto, return_tensors="pt", add_special_tokens=False).to("cuda")

    saida = model.generate(
        **entrada,
        max_new_tokens=max_novos_tokens,
        do_sample=False,
    )

    # cortar o prompt, decodificar só o que é novo
    novos = saida[0][entrada["input_ids"].shape[1]:]
    return tk.decode(novos, skip_special_tokens=True)


In [10]:
with open("ft-llm-tool-calling/dados/teste.jsonl", encoding="utf-8") as f:
    testes = [json.loads(linha) for linha in f]

com_tool = [t for t in testes if len(t["messages"]) == 5]
sem_tool = [t for t in testes if len(t["messages"]) == 3]

print(f"{len(testes)} testes  |  {len(com_tool)} com tool  |  {len(sem_tool)} sem tool")

51 testes  |  37 com tool  |  14 sem tool


In [7]:
exemplo = com_tool[0]
mensagens = exemplo["messages"]

entrada = mensagens[:2]        # só system + user; cortamos tudo que o assistente disse

print("PERGUNTA")
print(entrada[1]["content"])

print("\n" + "="*70)
print("ESPERADO (o que estava no dataset)")
print(mensagens[2]["content"])

print("\n" + "="*70)
print("GERADO (o que o seu modelo produziu)")
print(responder(entrada))

PERGUNTA
Posso ter um orçamento para as minhas despesas?

ESPERADO (o que estava no dataset)
<tool_call>
{"nome_tool": "create_budget", "argumentos": {"income": 5000.0, "expenses": [{"item": "Aluguel", "valor": 2000.0}, {"item": "Alimenta\u00e7\u00e3o", "valor": 800.0}, {"item": "Transporte", "valor": 400.0}, {"item": "Lazer", "valor": 300.0}]}}
</tool_call>

GERADO (o que o seu modelo produziu)
<tool_call>
{"nome_tool": "create_budget", "argumentos": {"income": 5000, "expenses": [{"descricao": "Aluguel", "valor": 1500}, {"descricao": "Alimentação", "valor": 800}, {"descricao": "Transporte", "valor": 300}, {"descricao": "Lazer", "valor": 400}]}}
</tool_call>


In [ ]:
exemplo = sem_tool[0]
mensagens = exemplo["messages"]
entrada = mensagens[:2]

print("PERGUNTA")
print(entrada[1]["content"])

print("\n" + "="*70)
print("ESPERADO")
print(mensagens[2]["content"])

print("\n" + "="*70)
print("GERADO")
print(responder(entrada))

In [11]:
import re

def extrair(saida):
    """Devolve (tipo, conteudo) a partir da saída bruta do modelo."""
    tool  = re.search(r"<tool_call>\s*(.*?)\s*</tool_call>",       saida, re.DOTALL)
    final = re.search(r"<final_answer>\s*(.*?)\s*</final_answer>", saida, re.DOTALL)

    if tool and final:
        return "ambos", saida
    if tool:
        return "tool_call", tool.group(1)
    if final:
        return "final_answer", final.group(1)
    return "sem_tag", saida


In [ ]:
import time

saidas_ft = []
inicio = time.time()

for n, ex in enumerate(testes, start=1):
    saidas_ft.append(responder(ex["messages"][:2]))
    if n % 10 == 0 or n == len(testes):
        print(f"{n}/{len(testes)}  —  {time.time() - inicio:.0f}s")

with open("saidas_ft.json", "w", encoding="utf-8") as f:
    json.dump(saidas_ft, f, ensure_ascii=False, indent=2)

print("salvo")

10/51  —  152s
20/51  —  356s
30/51  —  541s
40/51  —  710s
50/51  —  867s
51/51  —  917s
salvo


In [ ]:
for i in [0, 1, 2]:
    print(f"--- {i} | esperado: {'tool' if len(testes[i]['messages'])==5 else 'sem tool'}")
    print(saidas_ft[i][:400])
    print()


--- 0 | esperado: sem tool
<final_answer>
Melhorar as habilidades de resolução de conflitos envolve desenvolver comunicação eficaz, empatia e habilidades de negociação. Isso inclui aprender a ouvir ativamente as necessidades e perspectivas dos outros, a buscar soluções mutuamente benéficas e a manter a calma em situações tensas. Além disso, a capacidade de identificar e abordar as causas raiz dos conflitos é fundamental par

--- 1 | esperado: sem tool
<final_answer>
Melhorar as habilidades de gerenciamento de conflitos envolve desenvolver comunicação eficaz, empatia e habilidades de negociação. Isso inclui aprender a ouvir ativamente as necessidades e perspectivas dos outros, a buscar soluções mutuamente benéficas e a manter a calma em situações tensas. Além disso, a capacidade de resolver conflitos de forma construtiva e respeitosa é fundament

--- 2 | esperado: sem tool
<final_answer>
A negociação e a barganha são conceitos relacionados, mas possuem diferenças importantes em sua natu

In [ ]:
tk = getattr(tokenizer, "tokenizer", tokenizer)

for nome, saidas in [("original", saidas_ft)]:
    truncadas = [i for i, s in enumerate(saidas)
                 if extrair(s)[0] == "sem_tag"]
    tokens = [len(tk(s)["input_ids"]) for s in saidas]
    print(f"{nome}: sem_tag em {len(truncadas)} casos {truncadas}")
    print(f"  tokens gerados — máximo: {max(tokens)}, no teto (300): {sum(t >= 299 for t in tokens)}")


original: sem_tag em 1 casos [28]
  tokens gerados — máximo: 300, no teto (300): 1


In [ ]:
saidas_ft[28] = responder(testes[28]["messages"][:2], max_novos_tokens=800)

print("tipo:", extrair(saidas_ft[28])[0])
print("tokens:", len(tk(saidas_ft[28])["input_ids"]))
print()
print(saidas_ft[28])

with open("saidas_ft.json", "w", encoding="utf-8") as f:
    json.dump(saidas_ft, f, ensure_ascii=False, indent=2)


tipo: final_answer
tokens: 404

<final_answer>
Lidar com críticas e avaliações de forma construtiva é fundamental para o crescimento pessoal e profissional. A chave é aprender a ouvir, analisar e responder de maneira madura e respeitosa.

Aqui estão alguns passos práticos para lidar com críticas e avaliações:

1. **Mantenha a calma e evite reações defensivas:** Quando receber uma crítica, a primeira reação pode ser ficar na defensiva. Respire fundo e tente ouvir o ponto de vista da pessoa sem interromper ou se justificar imediatamente.
2. **Ouça ativamente e peça esclarecimentos:** Entenda exatamente o que está sendo dito. Pergunte "Você poderia me dar um exemplo específico?" ou "O que exatamente você acha que poderia ser melhorado?". Isso ajuda a entender a crítica de forma clara.
3. **Avalie a crítica objetivamente:** Separe o que é construtivo do que é apenas opinião ou ataque pessoal. Concentre-se nos pontos que podem ser melhorados e ignore os comentários destrutivos ou mal-intenc

In [13]:
import random, collections

def embaralhar_ferramentas(sp, semente):
    m = re.search(r"<ferramentas>\n(.*?)</ferramentas>", sp, re.DOTALL)
    linhas = [l for l in m.group(1).split("\n") if l.strip()]
    random.Random(semente).shuffle(linhas)
    novo = "<ferramentas>\n" + "\n".join(linhas) + "\n</ferramentas>"
    return sp[:m.start()] + novo + sp[m.end():]


entradas_shuf = [
    [{"role": "system", "content": embaralhar_ferramentas(ex["messages"][0]["content"], i)},
     ex["messages"][1]]
    for i, ex in enumerate(testes)
]

def nomes_das_tools(sp):
    corpo = re.search(r"<ferramentas>\n(.*?)</ferramentas>", sp, re.DOTALL).group(1)
    return sorted(l.split(",")[0][5:] for l in corpo.split("\n") if l.strip())

print("mesmo conjunto de tools em todos:", all(
    nomes_das_tools(ex["messages"][0]["content"]) == nomes_das_tools(e[0]["content"])
    for ex, e in zip(testes, entradas_shuf)))

pos = collections.Counter()
for ex, e in zip(testes, entradas_shuf):
    if len(ex["messages"]) != 5:
        continue
    corpo = re.search(r"<ferramentas>\n(.*?)</ferramentas>", e[0]["content"], re.DOTALL).group(1)
    ordem = [l.split(",")[0][5:] for l in corpo.split("\n") if l.strip()]
    _, gab = extrair(ex["messages"][2]["content"])
    pos[ordem.index(json.loads(gab)["nome_tool"]) + 1] += 1

print("posição da tool correta — antes:  {4: 37}")
print("posição da tool correta — agora: ", dict(sorted(pos.items())))

i_ex = next(i for i, ex in enumerate(testes) if len(ex["messages"]) == 5)
print(f"\n--- exemplo {i_ex}, ANTES ---")
print(re.search(r"<ferramentas>\n(.*?)</ferramentas>", testes[i_ex]["messages"][0]["content"], re.DOTALL).group(1))
print("--- DEPOIS ---")
print(re.search(r"<ferramentas>\n(.*?)</ferramentas>", entradas_shuf[i_ex][0]["content"], re.DOTALL).group(1))


mesmo conjunto de tools em todos: True
posição da tool correta — antes:  {4: 37}
posição da tool correta — agora:  {1: 12, 2: 8, 3: 10, 4: 7}

--- exemplo 3, ANTES ---
nome:get_user_info, descrição: Retorna informações de um usuário específico., parâmetros: [{'nome': 'user_id', 'tipo': 'integer', 'obrigatorio': True}]
nome:generate_report, descrição: Gera um relatório com base nos dados especificados., parâmetros: [{'nome': 'data', 'tipo': 'array', 'obrigatorio': True}, {'nome': 'format', 'tipo': 'string', 'obrigatorio': False}]
nome:get_traffic_info, descrição: Retorna informações de trânsito para uma rota específica., parâmetros: [{'nome': 'origin', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'destination', 'tipo': 'string', 'obrigatorio': True}]
nome:create_budget, descrição: Cria um orçamento com base nas receitas e despesas., parâmetros: [{'nome': 'income', 'tipo': 'number', 'obrigatorio': True}, {'nome': 'expenses', 'tipo': 'array', 'obrigatorio': True}]

--- DEPOIS ---
nome

In [ ]:
saidas_shuf = []
inicio = time.time()

for n, entrada in enumerate(entradas_shuf, start=1):
    saidas_shuf.append(responder(entrada, max_novos_tokens=800))
    if n % 10 == 0 or n == len(entradas_shuf):
        print(f"{n}/{len(entradas_shuf)}  —  {time.time() - inicio:.0f}s")

with open("saidas_shuf.json", "w", encoding="utf-8") as f:
    json.dump(saidas_shuf, f, ensure_ascii=False, indent=2)
print("salvo")


10/51  —  150s
20/51  —  340s
30/51  —  531s
40/51  —  686s
50/51  —  840s
51/51  —  901s
salvo


In [14]:
tools_catalogo = json.load(open("ft-llm-tool-calling/dados/tools_catalogo.json", encoding="utf-8"))
indice_tools = {t["nome"]: t for t in tools_catalogo}

MAPA_TIPOS = {"string": str, "number": (int, float), "integer": int,
              "boolean": bool, "array": list, "object": dict}

def avaliar(exemplo, saida):
    esperado = "tool_call" if len(exemplo["messages"]) == 5 else "final_answer"
    tipo, conteudo = extrair(saida)

    r = {"esperado": esperado, "obtido": tipo,
         "decidiu_certo": tipo == esperado,
         "json_valido": None, "tool_certa": None,
         "params_ok": None, "tipos_ok": None}

    if esperado != "tool_call" or tipo != "tool_call":
        return r

    try:
        chamada = json.loads(conteudo)
    except json.JSONDecodeError:
        r["json_valido"] = False
        return r
    r["json_valido"] = True

    _, gabarito = extrair(exemplo["messages"][2]["content"])
    nome_certo  = json.loads(gabarito)["nome_tool"]
    nome_obtido = chamada.get("nome_tool")
    r["tool_certa"] = nome_obtido == nome_certo

    schema = indice_tools.get(nome_obtido)
    if schema is None:
        r["params_ok"] = False
        return r

    args = chamada.get("argumentos")
    if not isinstance(args, dict):
        r["params_ok"] = False
        return r

    permitidos   = {p["nome"] for p in schema["parametros"]}
    obrigatorios = {p["nome"] for p in schema["parametros"] if p["obrigatorio"]}
    r["params_ok"] = obrigatorios <= args.keys() <= permitidos

    r["tipos_ok"] = all(
        isinstance(args[p["nome"]], MAPA_TIPOS.get(p["tipo"], object))
        for p in schema["parametros"] if p["nome"] in args
    )
    return r


#### Carregar os JSONs salvos

In [15]:
saidas_ft   = json.load(open("saidas_ft.json",   encoding="utf-8"))
saidas_shuf = json.load(open("saidas_shuf.json", encoding="utf-8"))
saidas_base = json.load(open("saidas_base.json", encoding="utf-8"))

tk = getattr(tokenizer, "tokenizer", tokenizer)

print(len(saidas_ft), len(saidas_shuf), len(saidas_base))   # 51 51 51

51 51 51


In [16]:
CAMPOS = ["decidiu_certo", "json_valido", "tool_certa", "params_ok", "tipos_ok"]

def resumir(nome, saidas):
    res = [avaliar(ex, s) for ex, s in zip(testes, saidas)]
    print(f"\n### {nome}")
    for campo in CAMPOS:
        v = [r[campo] for r in res if r[campo] is not None]
        print(f"  {campo:15} {f'{sum(v)}/{len(v)} = {sum(v)/len(v):.0%}' if v else '—'}")
    print("  obtido:", dict(collections.Counter(r["obtido"] for r in res)))
    return res

res_ft   = resumir("A — original (tool correta sempre no fim)", saidas_ft)
res_shuf = resumir("B — ordem embaralhada", saidas_shuf)

print("\n=== falhas, com a pergunta ===")
for nome, res in [("A", res_ft), ("B", res_shuf)]:
    for i, r in enumerate(res):
        ruins = [k for k in CAMPOS if r[k] is False]
        if ruins:
            print(f"[{nome}] {i:3} {ruins}")
            print(f"          {testes[i]['messages'][1]['content'][:75]}")



### A — original (tool correta sempre no fim)
  decidiu_certo   51/51 = 100%
  json_valido     37/37 = 100%
  tool_certa      37/37 = 100%
  params_ok       37/37 = 100%
  tipos_ok        37/37 = 100%
  obtido: {'final_answer': 14, 'tool_call': 37}

### B — ordem embaralhada
  decidiu_certo   49/51 = 96%
  json_valido     34/35 = 97%
  tool_certa      34/34 = 100%
  params_ok       34/34 = 100%
  tipos_ok        34/34 = 100%
  obtido: {'final_answer': 14, 'tool_call': 35, 'sem_tag': 2}

=== falhas, com a pergunta ===
[B]  15 ['json_valido']
          Eu quero criar uma fatura para um cliente que pagou com cheque
[B]  26 ['decidiu_certo']
          Quero saber o que esses dados significam.
[B]  36 ['decidiu_certo']
          Necessito uma ajuda para organizar as minhas finanças.


In [ ]:
for i in [15, 26, 36]:
    print("="*70)
    print(f"[{i}] {testes[i]['messages'][1]['content']}")
    print(f"esperado: {'tool_call' if len(testes[i]['messages'])==5 else 'final_answer'}")
    print(f"\n--- A ({extrair(saidas_ft[i])[0]}, {len(tk(saidas_ft[i])['input_ids'])} tokens)")
    print(saidas_ft[i][:600])
    print(f"\n--- B ({extrair(saidas_shuf[i])[0]}, {len(tk(saidas_shuf[i])['input_ids'])} tokens)")
    print(saidas_shuf[i][:900])

[15] Eu quero criar uma fatura para um cliente que pagou com cheque
esperado: tool_call

--- A (tool_call, 170 tokens)
<tool_call>
{"nome_tool": "generate_invoice", "argumentos": {"items": [{"descricao": "Servi\u00e3o de consultoria", "quantidade": 1, "preco_unitario": 500.0, "unidade": "R$"}, {"descricao": "Imposto sobre venda", "quantidade": 1, "preco_unitario": 50.0, "unidade": "R$"}], "customer_info": {"nome": "Jo\u00e3o Silva", "email": "joao.silva@exemplo.com", "endereco": "Rua das Flores, 123, S\u00e3o Paulo, SP"}}}
</tool_call>

--- B (tool_call, 116 tokens)
<tool_call>
{"nome_tool": "generate_invoice", "argumentos": {"items": [{"descricao": "Servi\u00e3o de consultoria", "quantidade": 1, "preco_unitario": 500.00}], "customer_info": {"nome": "Cliente Exemplo", "email": "cliente@exemplo.com", "endereco": "Rua das Flores, 123, S\u00e3o Paulo, SP"}}
</tool_call>
[26] Quero saber o que esses dados significam.
esperado: tool_call

--- A (tool_call, 67 tokens)
<tool_call>
{"nome_tool

In [ ]:
i = 3   # um caso com uma tool conhecida
entrada = testes[i]["messages"][:2]

with model.disable_adapter():
    teste_base = responder(entrada, max_novos_tokens=800)

print("PERGUNTA:", entrada[1]["content"])
print("\n--- COM adaptador (o seu modelo) ---")
print(saidas_ft[i][:400])
print("\n--- SEM adaptador (base) ---")
print(teste_base[:400])
print("\nsaídas idênticas?", saidas_ft[i] == teste_base)


PERGUNTA: Posso ter um orçamento para as minhas despesas?

--- COM adaptador (o seu modelo) ---
<tool_call>
{"nome_tool": "create_budget", "argumentos": {"income": 5000, "expenses": [{"descricao": "Aluguel", "valor": 1500}, {"descricao": "Alimentação", "valor": 800}, {"descricao": "Transporte", "valor": 300}, {"descricao": "Lazer", "valor": 400}]}}
</tool_call>

--- SEM adaptador (base) ---
Para que eu possa criar um orçamento para suas despesas, preciso que você me forneça as informações sobre suas receitas (entradas de dinheiro) e suas despesas (saídas de dinheiro).

Você pode me dizer quanto você ganha e quais são suas despesas? Por exemplo:

*   **Receitas:** (Ex: Salário, renda extra)
*   **Despesas:** (Ex: Aluguel, contas de consumo, alimentação, lazer, etc.)

Assim que você me

saídas idênticas? False


In [ ]:
saidas_base = []
inicio = time.time()

with model.disable_adapter():
    for n, ex in enumerate(testes, start=1):
        saidas_base.append(responder(ex["messages"][:2], max_novos_tokens=800))
        if n % 10 == 0 or n == len(testes):
            print(f"{n}/{len(testes)}  —  {time.time() - inicio:.0f}s")

with open("saidas_base.json", "w", encoding="utf-8") as f:
    json.dump(saidas_base, f, ensure_ascii=False, indent=2)
print("salvo")


10/51  —  376s
20/51  —  659s
30/51  —  860s
40/51  —  1104s
50/51  —  1393s
51/51  —  1478s
salvo


In [ ]:
res_base = resumir("C — modelo base, sem adaptador", saidas_base)

print("\n=== comparação ===")
for nome, res in [("A original ", res_ft), ("B embaralh.", res_shuf), ("C base     ", res_base)]:
    linha = []
    for campo in CAMPOS:
        v = [r[campo] for r in res if r[campo] is not None]
        linha.append(f"{campo[:8]}={sum(v)}/{len(v)}" if v else f"{campo[:8]}=—")
    print(f"{nome} | " + "  ".join(linha))

print("\n=== o que cada condição produziu ===")
for nome, res in [("A", res_ft), ("B", res_shuf), ("C", res_base)]:
    print(f"{nome}: {dict(collections.Counter(r['obtido'] for r in res))}")

tokens_c = [len(tk(s)["input_ids"]) for s in saidas_base]
print(f"\nC — tokens: mediana {sorted(tokens_c)[len(tokens_c)//2]}, "
      f"máximo {max(tokens_c)}, no teto (800): {sum(t >= 799 for t in tokens_c)}")



### C — modelo base, sem adaptador
  decidiu_certo   27/51 = 53%
  json_valido     17/17 = 100%
  tool_certa      17/17 = 100%
  params_ok       17/17 = 100%
  tipos_ok        17/17 = 100%
  obtido: {'sem_tag': 21, 'final_answer': 12, 'tool_call': 17, 'ambos': 1}

=== comparação ===
A original  | decidiu_=51/51  json_val=37/37  tool_cer=37/37  params_o=37/37  tipos_ok=37/37
B embaralh. | decidiu_=49/51  json_val=34/35  tool_cer=34/34  params_o=34/34  tipos_ok=34/34
C base      | decidiu_=27/51  json_val=17/17  tool_cer=17/17  params_o=17/17  tipos_ok=17/17

=== o que cada condição produziu ===
A: {'final_answer': 14, 'tool_call': 37}
B: {'final_answer': 14, 'tool_call': 35, 'sem_tag': 2}
C: {'sem_tag': 21, 'final_answer': 12, 'tool_call': 17, 'ambos': 1}

C — tokens: mediana 60, máximo 800, no teto (800): 1


In [ ]:
PARSEAVEL = {"tool_call", "final_answer"}

for nome, res in [("A", res_ft), ("B", res_shuf), ("C", res_base)]:
    formatou = [r for r in res if r["obtido"] in PARSEAVEL]
    certos   = [r for r in formatou if r["decidiu_certo"]]
    print(f"{nome}  formatou: {len(formatou)}/{len(res)} = {len(formatou)/len(res):>4.0%}"
          f"   |   decidiu certo dado que formatou: {len(certos)}/{len(formatou)} = {len(certos)/len(formatou):.0%}")


A  formatou: 51/51 = 100%   |   decidiu certo dado que formatou: 51/51 = 100%
B  formatou: 49/51 =  96%   |   decidiu certo dado que formatou: 49/49 = 100%
C  formatou: 29/51 =  57%   |   decidiu certo dado que formatou: 27/29 = 93%


In [18]:
def inverter_contagem(sp, tem_tool, nome_correta, semente):
    m = re.search(r"<ferramentas>\n(.*?)</ferramentas>", sp, re.DOTALL)
    linhas = [l for l in m.group(1).split("\n") if l.strip()]
    rng = random.Random(semente)

    if tem_tool:
        # remove uma distratora: qualquer linha que NÃO seja a da tool correta
        candidatas = [l for l in linhas if not l.startswith(f"nome:{nome_correta},")]
        linhas.remove(rng.choice(candidatas))
    else:
        # acrescenta uma distratora no fim
        ja_listadas = {l.split(",")[0][5:] for l in linhas}
        disponiveis = [t for nome, t in indice_tools.items() if nome not in ja_listadas]
        t = rng.choice(disponiveis)
        linhas.append(f'nome:{t["nome"]}, descrição: {t["descricao"]}, parâmetros: {t["parametros"]}')

    novo = "<ferramentas>\n" + "\n".join(linhas) + "\n</ferramentas>"
    return sp[:m.start()] + novo + sp[m.end():]


entradas_cont = []
for i, ex in enumerate(testes):
    tem_tool = len(ex["messages"]) == 5
    if tem_tool:
        _, gab = extrair(ex["messages"][2]["content"])
        nome_correta = json.loads(gab)["nome_tool"]
    else:
        nome_correta = None
    entradas_cont.append([
        {"role": "system",
         "content": inverter_contagem(ex["messages"][0]["content"], tem_tool, nome_correta, i)},
        ex["messages"][1],
    ])


# --- verificação 1: a contagem inverteu ---
cont = collections.Counter()
for ex, e in zip(testes, entradas_cont):
    tipo = "com_tool" if len(ex["messages"]) == 5 else "sem_tool"
    cont[(tipo, len(nomes_das_tools(e[0]["content"])))] += 1
print("(tipo, nº de tools) ->", dict(cont))
print("  esperado: {('com_tool', 3): 37, ('sem_tool', 4): 14}")

# --- verificação 2: a tool correta NÃO sumiu ---
faltando = []
for i, (ex, e) in enumerate(zip(testes, entradas_cont)):
    if len(ex["messages"]) != 5:
        continue
    _, gab = extrair(ex["messages"][2]["content"])
    if json.loads(gab)["nome_tool"] not in nomes_das_tools(e[0]["content"]):
        faltando.append(i)
print("casos onde a tool correta sumiu:", faltando, " — esperado: []")

# --- verificação 3: olhar a linha acrescentada ---
i_sem = next(i for i, ex in enumerate(testes) if len(ex["messages"]) == 3)
print(f"\n--- exemplo {i_sem} (sem_tool), agora com 4 tools ---")
print(re.search(r"<ferramentas>\n(.*?)</ferramentas>",
                entradas_cont[i_sem][0]["content"], re.DOTALL).group(1))


(tipo, nº de tools) -> {('sem_tool', 4): 14, ('com_tool', 3): 37}
  esperado: {('com_tool', 3): 37, ('sem_tool', 4): 14}
casos onde a tool correta sumiu: []  — esperado: []

--- exemplo 0 (sem_tool), agora com 4 tools ---
nome:get_weather_alerts, descrição: Retorna alertas de condições climáticas adversas., parâmetros: [{'nome': 'location', 'tipo': 'string', 'obrigatorio': True}]
nome:manage_financial_goals, descrição: Gerencia metas financeiras com base nas economias e investimentos., parâmetros: [{'nome': 'goals', 'tipo': 'array', 'obrigatorio': True}]
nome:analyze_data, descrição: Analisa um conjunto de dados e retorna estatísticas., parâmetros: [{'nome': 'data', 'tipo': 'array', 'obrigatorio': True}, {'nome': 'metrics', 'tipo': 'array', 'obrigatorio': True}]
nome:track_workout, descrição: Rastreia as atividades físicas realizadas., parâmetros: [{'nome': 'exercise', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'duration', 'tipo': 'integer', 'obrigatorio': True}]



In [21]:
import time

saidas_cont = []
inicio = time.time()

for n, entrada in enumerate(entradas_cont, start=1):
    saidas_cont.append(responder(entrada, max_novos_tokens=800))
    if n % 10 == 0 or n == len(entradas_cont):
        print(f"{n}/{len(entradas_cont)}  —  {time.time() - inicio:.0f}s")

with open("saidas_cont.json", "w", encoding="utf-8") as f:
    json.dump(saidas_cont, f, ensure_ascii=False, indent=2)
print("salvo")


10/51  —  147s
20/51  —  342s
30/51  —  546s
40/51  —  717s
50/51  —  871s
51/51  —  927s
salvo


In [22]:
res_cont = resumir("D — contagem invertida", saidas_cont)

print("\n=== decidiu_certo por tipo (o que realmente importa aqui) ===")
for rotulo, n_msgs in [("com_tool  4→3", 5), ("sem_tool  3→4", 3)]:
    v = [r["decidiu_certo"] for ex, r in zip(testes, res_cont) if len(ex["messages"]) == n_msgs]
    print(f"  {rotulo:15} {sum(v)}/{len(v)} = {sum(v)/len(v):.0%}   (em A era 100%)")

print("\n=== falhas ===")
for i, r in enumerate(res_cont):
    ruins = [k for k in CAMPOS if r[k] is False]
    if ruins:
        print(f"{i:3} {ruins}  |  {testes[i]['messages'][1]['content'][:65]}")


### D — contagem invertida
  decidiu_certo   51/51 = 100%
  json_valido     36/37 = 97%
  tool_certa      36/36 = 100%
  params_ok       36/36 = 100%
  tipos_ok        36/36 = 100%
  obtido: {'final_answer': 14, 'tool_call': 37}

=== decidiu_certo por tipo (o que realmente importa aqui) ===
  com_tool  4→3   37/37 = 100%   (em A era 100%)
  sem_tool  3→4   14/14 = 100%   (em A era 100%)

=== falhas ===
 15 ['json_valido']  |  Eu quero criar uma fatura para um cliente que pagou com cheque
